# Clase 019 — Ordenamiento y búsqueda

**Parte 0** · VanderPlas cap. 2 § 2.8.

> 🎯 sort vs argsort, top-K con partition, búsqueda binaria con searchsorted.

> ⏱️ ~60 min

## ⚙️ Setup

In [ ]:
import numpy as np
import time
rng = np.random.default_rng(42)

## 1️⃣ `np.sort` vs `arr.sort()`

- `np.sort(arr)` → **devuelve copia** ordenada
- `arr.sort()` → ordena **in-place**, devuelve None

In [ ]:
arr = rng.integers(0, 100, 10)
print(f'original   : {arr}')
print(f'np.sort    : {np.sort(arr)}')
print(f'original   : {arr}  ← sin cambios')

arr.sort()
print(f'tras .sort(): {arr}  ← mutado in-place')

## 2️⃣ `argsort` — el truco del top-K

`argsort` devuelve los **índices que ordenarían** el array. Aplicarlos al array original devuelve ordenado:

```python
idx = arr.argsort()
ordenado = arr[idx]
```

Para top-K (los K más grandes): `arr[arr.argsort()][-K:]`.

In [ ]:
puntajes = rng.integers(0, 1000, 20)
print(f'puntajes: {puntajes}')

idx_orden = puntajes.argsort()
print(f'top-5    : {puntajes[idx_orden[-5:]]}')
print(f'bottom-5 : {puntajes[idx_orden[:5]]}')

## 3️⃣ Ranking — argsort de argsort

Para obtener el **ranking** (posición 1..N de cada elemento):

In [ ]:
ranking = puntajes.argsort().argsort() + 1   # +1 para 1-indexed
for p, r in zip(puntajes, ranking):
    print(f'puntaje={p:4d}  ranking={r}')

## 4️⃣ Ordenamiento por eje en matrices

In [ ]:
M = rng.integers(0, 100, (4, 5))
print('Original:')
print(M)

print('\nOrdenado por filas (axis=1):')
print(np.sort(M, axis=1))

print('\nOrdenado por columnas (axis=0):')
print(np.sort(M, axis=0))

## 5️⃣ `np.partition` — top-K más rápido

Si solo quieres los K más chicos/grandes, **no necesitas ordenar todo el array**. `partition(arr, K)` deja los K menores en las primeras K posiciones (no necesariamente ordenados entre sí), y el resto después.

Complejidad: O(n) vs O(n log n) del sort completo.

In [ ]:
N = 1_000_000
K = 100
datos = rng.normal(0, 1, N)

# Sort completo
t0 = time.perf_counter()
top_sort = np.sort(datos)[-K:]
t1 = time.perf_counter()

# Partition
t2 = time.perf_counter()
top_part = np.partition(datos, -K)[-K:]
top_part.sort()   # opcional: ordenar solo esos K
t3 = time.perf_counter()

print(f'sort completo : {(t1-t0)*1000:.1f} ms')
print(f'partition     : {(t3-t2)*1000:.1f} ms')
print(f'speedup       : {(t1-t0)/(t3-t2):.1f}×')
print(f'mismo top?    : {np.array_equal(np.sort(top_sort), np.sort(top_part))}')

## 6️⃣ `np.searchsorted` — búsqueda binaria O(log n)

En un array ordenado, encuentra el índice donde insertar un valor para mantenerlo ordenado:

In [ ]:
ordenado = np.array([10, 20, 30, 40, 50, 60, 70, 80, 90])

# Dónde se insertaría 35
print(np.searchsorted(ordenado, 35))    # 3

# Batch: múltiples valores
print(np.searchsorted(ordenado, [5, 35, 75, 100]))

**Uso: calcular percentil de un valor**:

```python
rank = np.searchsorted(arr_ordenado, valor)
percentil = 100 * rank / len(arr_ordenado)
```

In [ ]:
# Percentil de un puntaje en una distribución
puntajes = np.sort(rng.normal(70, 10, 10_000))
mi_puntaje = 85
rank = np.searchsorted(puntajes, mi_puntaje)
percentil = 100 * rank / len(puntajes)
print(f'puntaje {mi_puntaje} → percentil {percentil:.1f}')

## 7️⃣ `np.unique` — únicos y cuentas

In [ ]:
categorias = rng.choice(['A', 'B', 'C', 'D'], size=1000, p=[0.5, 0.3, 0.15, 0.05])
valores, cuentas = np.unique(categorias, return_counts=True)
for v, c in zip(valores, cuentas):
    print(f'{v}: {c:4d}  ({100*c/len(categorias):.1f}%)')

## ✅ Checklist

- [ ] Distingo `np.sort()` (copia) de `arr.sort()` (in-place)
- [ ] Sé usar `argsort` para top-K y rankings
- [ ] Ordeno matrices por axis
- [ ] Uso `partition` cuando solo necesito top-K
- [ ] Sé que `searchsorted` es O(log n)

## 📝 Homework

Ver `README.md`. Top-100 con benchmark partition vs sort, ranking, percentil con searchsorted, unique.

## 📖 Definiciones y características

**`np.sort` vs `arr.sort()`**

**`np.sort(arr)`** devuelve copia ordenada (no muta). **`arr.sort()`** ordena in-place y retorna None. Mismo patrón que `sorted(list)` vs `list.sort()`.

**`argsort`**

Devuelve los **índices** que ordenarían el array. `idx = arr.argsort(); ordenado = arr[idx]`. Base de top-K, rankings, alineación entre arrays correlacionados.

**`np.partition`**

Quick-select O(n): garantiza que los K menores quedan en las primeras K posiciones (no necesariamente ordenados entre sí), el resto después. Mucho más rápido que sort completo cuando solo necesitas top-K.

**`np.searchsorted`**

Búsqueda binaria O(log n) en array ordenado. Devuelve el índice donde insertar un valor para mantener orden. Útil para calcular percentiles, bucketing.

**`np.unique`**

Devuelve únicos ordenados. Con `return_counts=True` devuelve también las frecuencias — alternativa rápida a `Counter` para datos numéricos.

## ⚠️ Errores comunes

| Síntoma / mensaje | Causa y cómo arreglar |
|---|---|
| Ordeno con `arr.sort()` y la variable queda en `None` | `sort()` es in-place — modifica `arr` y retorna `None`. **Fix**: `ordenado = np.sort(arr)` (con `np.sort`). |
| Top-K con `sort()[-K:]` es lento para N grande, K chico | Sort completo es O(N log N). **Fix**: `np.partition(arr, -K)[-K:]` es O(N). Acopla con `.sort()` si necesitas los K ordenados internamente. |
| `argsort` da resultado raro con matriz | Sin `axis`, ordena cada fila/columna independiente según `axis=-1` por default. Para ordenar matriz completa por una columna, usa `arr[arr[:, col].argsort()]`. |
| `np.searchsorted` da índice fuera del array | Si el valor es mayor que todos, devuelve `len(arr)`. Es el comportamiento correcto ("insertar al final"). **Fix**: clipea con `np.clip(idx, 0, len(arr)-1)` si vas a indexar. |
| `np.unique(arr_2d)` aplana el array | Por default, `unique` trabaja sobre array aplanado. Para únicos por fila/columna: `unique(arr, axis=0)`. |

## ❓ Preguntas frecuentes

**❓ ¿Cuándo `argsort` y cuándo `sort`?**

Si solo necesitas los valores ordenados, `sort`. Si necesitas el orden para **aplicarlo a otros arrays correlacionados** (ej: ordenar `nombres` por `puntajes`), `argsort` te da los índices.

**❓ ¿`partition` o `heapq.nlargest`?**

`partition` para arrays NumPy (vectorizado, C). `heapq.nlargest(K, lst)` para listas Python. Para K muy pequeño (3-5) sobre N grande, similares; partition gana en arrays grandes.

**❓ ¿Cómo ordeno por múltiples claves (lexicográfico)?**

`np.lexsort([clave_secundaria, clave_principal])` — devuelve índices. Atención: el orden es **al revés** (último argumento = key primaria).

**❓ ¿`np.unique` preserva el orden de primera aparición?**

**No** — siempre ordena. Para preservar orden de aparición: `_, idx = np.unique(arr, return_index=True); arr[np.sort(idx)]`.

**❓ ¿Existe equivalente a SQL `ORDER BY x DESC`?**

`arr[arr.argsort()][::-1]` o `arr[arr.argsort()[::-1]]`. NumPy no tiene flag `reverse=` como `sorted()` Python — invierte tú.

## 🔗 Referencias

- VanderPlas cap. 2 § 2.8
- [Sorting reference](https://numpy.org/doc/stable/reference/routines.sort.html)

➡️ **Siguiente:** [020 — Álgebra lineal](../020-numpy-algebra-lineal-con-numpy-linalg/README.md)

## ✅ Soluciones de los ejercicios

A continuación, cada ejercicio de la sección `🧪 Ejercicios` del README resuelto y comentado. Todo el código es **ejecutable sin conexión** (datos sintéticos) e incluye `assert`/`print` para que compruebes el resultado. Intenta resolverlos por tu cuenta antes de mirar la solución.

**Ej. 1 — Top-10 de 1M de puntajes:** `sort` vs `partition`.

In [ ]:
import numpy as np
rng = np.random.default_rng(19)
puntajes = rng.integers(0, 1_000_000, size=1_000_000)
top_sort = np.sort(puntajes)[-10:]              # ordena TODO y toma los 10 ultimos
top_part = np.partition(puntajes, -10)[-10:]    # solo garantiza los 10 mayores (mas rapido)
assert np.array_equal(np.sort(top_part), top_sort)
print('Top-10:', top_sort)
print('partition da los mismos 10 valores sin ordenar el resto.')

**Ej. 2 — Ranking con `argsort`** (1 = mejor).

In [ ]:
notas = np.array([88, 72, 95, 60, 81])
orden = np.argsort(-notas)                 # indices de mayor a menor
ranking = np.empty_like(orden)
ranking[orden] = np.arange(1, len(notas) + 1)
print('notas  :', notas)
print('ranking:', ranking)
assert ranking[np.argmax(notas)] == 1

**Ej. 3 — Ordena cada columna** de una matriz 10x5.

In [ ]:
M = rng.integers(0, 100, size=(10, 5))
M_sorted = np.sort(M, axis=0)              # ordena cada columna de forma independiente
assert (np.diff(M_sorted, axis=0) >= 0).all()   # cada columna queda no-decreciente
print('Primera columna ordenada:', M_sorted[:, 0])

**Ej. 4 — Percentil con `searchsorted`.**

In [ ]:
def percentil_de(v, arr_ordenado):
    pos = np.searchsorted(arr_ordenado, v)     # cuantos elementos son < v
    return 100.0 * pos / len(arr_ordenado)

datos = np.sort(rng.normal(50, 10, 1000))
p = percentil_de(50, datos)
print(f'El valor 50 esta aprox. en el percentil {p:.1f}')
assert 30 < p < 70                             # ~ percentil 50 (la media)

**Ej. 5 — `np.unique` con cuentas.**

In [ ]:
cats = rng.choice(['A', 'B', 'C'], size=200, p=[0.5, 0.3, 0.2])
valores, cuentas = np.unique(cats, return_counts=True)
for v, c in zip(valores, cuentas):
    print(f'{v}: {c}')
assert cuentas.sum() == 200